# Point to Point Link
---


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import xarray as xr
from hics import HCS, HICSLogger


from hics.plotting import view_surface_profile, viewcs, plotnlcd
from scipy.spatial.transform import Rotation

from xant import ureg, XANTLogger
from xant.antenna import common
from xant.propagation import rflink


XANTLogger.unmute()
XANTLogger.level = "DEBUG"
# pv.set_jupyter_backend("trame")

In [ ]:
llatx = (39.998918, -105.28254, 21)
hcstx = HCS.from_crs((llatx[0] * ureg.degree, llatx[1] * ureg.degree, llatx[2] * ureg.m), hagl=True)

llarx = (39.993457, -105.264, 60)
hcsrx = HCS.from_crs((llarx[0] * ureg.degree, llarx[1] * ureg.degree, llarx[2] * ureg.m), hagl=True)


In [ ]:
mounttx = HCS(
    (0, 0, 0) * ureg.m,
    rotation=Rotation.from_euler("ZXZ", [0, 91, 111], degrees=True),
    reference=hcstx,
)

mountrx = HCS(
    (0, 0, 0) * ureg.m,
    rotation=Rotation.from_euler("ZXZ", [0, 89, 291], degrees=True),
    reference=hcsrx,
)

In [ ]:
pltter = viewcs(mounttx, hcstx, vector_length=200)
pltter = viewcs(mountrx, hcstx, ax=pltter, vector_length=200)
pltter.show()

In [ ]:
res = view_surface_profile(mounttx, mountrx, aspect=0.1)

In [ ]:
plotnlcd([(39.925472, -105.331491), (40.102960, -105.102838)])


In [ ]:
plotnlcd([llatx[:2], llarx[:2]])

In [ ]:
# Frequency and wavelength
f0 = 5.2 * ureg.GHz
lam0 = (ureg.speed_of_light / f0).to("cm")
f0 = [5.2] * ureg.GHz

# Create antenna objects to emulate the Ubiquiti airFiber 5XHD
eta = 0.8
ant0 = common.CircularAperture(lam0 * 6, f0, mounttx) * np.sqrt(eta)
ant1 = common.CircularAperture(lam0 * 6, f0, mountrx) * np.sqrt(eta)

In [ ]:
tx_power = 29 * ureg.dBm
rx_power, prop_loss, incident_pol, txcs, rxcs = rflink.calculate_spatial_link(
    ant0,
    tx_power,
    ant1,
    propagation="itm_rflink",
    clutter_method=rflink.propagators.ClutterMethods.ITURP1812,
    clutter_kwargs={"lc_skip_ind": 10},
)
rx_power

In [ ]:
prop_loss

In [ ]:
ax = rflink.view_link_horizon(
    ant0,
    ant1,
    rx_power,
    incident_pol,
    prop_loss,
    aspect=0.2,
)